In [0]:
%sql
CREATE CATALOG IF NOT EXISTS proyecto_ecommerce;

CREATE SCHEMA IF NOT EXISTS proyecto_ecommerce.staging;
CREATE SCHEMA IF NOT EXISTS proyecto_ecommerce.raw;
CREATE SCHEMA IF NOT EXISTS proyecto_ecommerce.bronze;
CREATE SCHEMA IF NOT EXISTS proyecto_ecommerce.silver;
CREATE SCHEMA IF NOT EXISTS proyecto_ecommerce.gold;

CREATE VOLUME IF NOT EXISTS proyecto_ecommerce.staging.fuente_externa;
CREATE VOLUME IF NOT EXISTS proyecto_ecommerce.raw.raw_files;

In [0]:
import json
import csv
import random
from datetime import datetime, timedelta

BASE_PATH = "/Volumes/proyecto_ecommerce/staging/fuente_externa"
REFERENCIA_PATH = f"{BASE_PATH}/referencia"

CLIENTES_PATH = f"{REFERENCIA_PATH}/clientes.csv"
PRODUCTOS_PATH = f"{REFERENCIA_PATH}/productos.json"

with open(PRODUCTOS_PATH) as f:
    catalogo = json.load(f)
productos = catalogo["productos"]

with open(CLIENTES_PATH) as f:
    reader = csv.DictReader(f)
    clientes = [row["cliente_id"] for row in reader]

print(f"Productos disponibles: {len(productos)}")
print(f"Clientes disponibles: {len(clientes)}")
print("Ejemplo de producto:", productos[0])

In [0]:
def generar_orden_limpia(numero_secuencial: int) -> dict:
    """Genera una orden 'ideal', sin errores todavía."""

    cliente_id = random.choice(clientes)
    producto = random.choice(productos)

    cantidad = random.randint(1, 3)
    descuento_pct = random.choice([0, 0, 0, 0.10, 0.15])  # la mayoría sin descuento
    canal = random.choice(["app", "web", "marketplace"])
    estado = random.choices(
        ["completada", "cancelada", "pendiente"],
        weights=[0.85, 0.10, 0.05]  # la mayoría completadas, como en un negocio real
    )[0]

    # Timestamp aleatorio dentro de la última hora, simulando que "acaba de pasar"
    ahora = datetime.now()
    minutos_atras = random.randint(0, 60)
    fecha_hora = ahora - timedelta(minutes=minutos_atras)

    return {
        "orden_id": f"ORD{ahora.strftime('%Y%m')}{numero_secuencial:06d}",
        "fecha_hora": fecha_hora.strftime("%Y-%m-%dT%H:%M:%S"),
        "cliente_id": cliente_id,
        "producto_id": producto["producto_id"],
        "cantidad": cantidad,
        "precio_unitario": producto["precio_referencia"],
        "descuento_pct": descuento_pct,
        "canal": canal,
        "estado": estado,
    }


# Prueba con una sola orden antes de generar el lote completo
orden_prueba = generar_orden_limpia(1)
print(orden_prueba)

In [0]:
CANTIDAD_ORDENES_NUEVAS = random.randint(50, 100)
print(f"Este lote va a tener {CANTIDAD_ORDENES_NUEVAS} órdenes")

lote_ordenes = [
    generar_orden_limpia(i + 1)
    for i in range(CANTIDAD_ORDENES_NUEVAS)
]

print(f"Órdenes generadas: {len(lote_ordenes)}")
print("Primeras 3 órdenes:")
for o in lote_ordenes[:3]:
    print(o)

In [0]:
def corromper_orden(orden: dict) -> dict:
    """Recibe una orden limpia y, con cierta probabilidad, le mete errores
    intencionales — replicando lo que ya vimos en la data histórica real."""

    orden = orden.copy()  # nunca modifiques el diccionario original directamente

    # 5% de probabilidad: fecha_hora como número epoch en vez de texto ISO
    # (esto ya lo vimos en ordenes_202601.json, es un problema real)
    if random.random() < 0.06:
        dt = datetime.fromisoformat(orden["fecha_hora"])
        orden["fecha_hora"] = int(dt.timestamp() * 1000)

    # 3% de probabilidad: canal en mayúsculas inconsistentes
    if random.random() < 0.10:
        orden["canal"] = orden["canal"].upper()

    # 2% de probabilidad: precio_unitario como texto en vez de número
    if random.random() < 0.10:
        orden["precio_unitario"] = str(orden["precio_unitario"])

    # 2% de probabilidad: cliente_id con un typo (referencia a un cliente
    # que en realidad no existe -- un caso real de integridad referencial rota)
    if random.random() < 0.03:
        orden["cliente_id"] = orden["cliente_id"] + "X"

    return orden


# Aplicar la corrupción a cada orden del lote
lote_ordenes_sucio = [corromper_orden(o) for o in lote_ordenes]

# Contar cuántas órdenes quedaron distintas al original, para verificar que sí funcionó
diferencias = sum(1 for limpia, sucia in zip(lote_ordenes, lote_ordenes_sucio) if limpia != sucia)
print(f"Órdenes modificadas por la corrupción: {diferencias} de {len(lote_ordenes)}")

In [0]:
# Celda 5: Duplicar algunas órdenes completas dentro del lote

def duplicar_ordenes(lote, porcentaje_duplicados=0.15):
    n_duplicados = max(1, int(len(lote) * porcentaje_duplicados))
    duplicados = random.sample(lote, n_duplicados)
    lote_con_duplicados = lote + duplicados
    random.shuffle(lote_con_duplicados)
    return lote_con_duplicados, n_duplicados

lote_ordenes_final, n_dup = duplicar_ordenes(lote_ordenes_sucio)
print(f"Duplicados agregados: {n_dup}")
print(f"Total de registros en el lote final: {len(lote_ordenes_final)}")

In [0]:
import os

DESTINO_ORDENES_PATH = "/Volumes/proyecto_ecommerce/staging/fuente_externa"

timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
nombre_archivo = f"ordenes_nuevas_{timestamp_str}.json"
ruta_completa = f"{DESTINO_ORDENES_PATH}/{nombre_archivo}"

with open(ruta_completa, "w", encoding="utf-8") as f:
    json.dump(lote_ordenes_final, f, indent=2, ensure_ascii=False)

print(f"Guardado: {ruta_completa}")
print(f"Total de órdenes en el archivo: {len(lote_ordenes_final)}")

In [0]:
import shutil
import os
from pathlib import Path
from datetime import datetime

STAGING_PATH = "/Volumes/proyecto_ecommerce/staging/fuente_externa"
REFERENCIA_PATH = "/Volumes/proyecto_ecommerce/staging/referencia"
PROCESADOS_PATH = f"{STAGING_PATH}/procesados"
RAW_PATH = "/Volumes/proyecto_ecommerce/raw/raw_files"

os.makedirs(PROCESADOS_PATH, exist_ok=True)

fecha_ingesta = datetime.now().strftime("%Y-%m-%d")
destino_particion = f"{RAW_PATH}/ingestion_date={fecha_ingesta}"
os.makedirs(destino_particion, exist_ok=True)

archivos_copiados = 0

# 1. Órdenes: copiar a Raw y mover a procesados (se drenan)
for archivo in Path(STAGING_PATH).glob("*.*"):
    if archivo.is_file():
        shutil.copy(archivo, f"{destino_particion}/{archivo.name}")
        shutil.move(str(archivo), f"{PROCESADOS_PATH}/{archivo.name}")
        archivos_copiados += 1
        print(f"Procesado (orden): {archivo.name}")

# 2. Referencia: solo copiar a Raw, nunca mover (se queda disponible siempre)
for archivo in Path(REFERENCIA_PATH).glob("*.*"):
    if archivo.is_file():
        shutil.copy(archivo, f"{destino_particion}/{archivo.name}")
        archivos_copiados += 1
        print(f"Procesado (referencia): {archivo.name}")

print(f"\nTotal copiado a Raw: {archivos_copiados} archivos")